## Pendahuluan

Data panel menggabungkan dimensi *cross-section* dan *time series*, memungkinkan kita mengontrol heterogenitas individu yang tidak terobservasi. Tutorial ini membahas estimasi model data panel di STATA secara lengkap.

### Notasi Model

$$
y_{it} = \alpha_i + \mathbf{x}_{it}'\boldsymbol{\beta} + \varepsilon_{it}
$$

- $\alpha_i$: efek individu (fixed atau random)
- $\mathbf{x}_{it}$: vektor variabel independen
- $\varepsilon_{it}$: error term idiosinkratik

## Setup Data

Kita gunakan dataset `nlswork` (National Longitudinal Survey) bawaan STATA.

```stata
* Load data
webuse nlswork, clear

* Lihat struktur
describe
summarize ln_wage age tenure hours

* Deklarasi data panel
xtset idcode year

* Ringkasan panel
xtdescribe
xtsum ln_wage age tenure hours
```

> **`xtsum`** memberikan dekomposisi between vs within variation — penting untuk memahami sumber variasi data.

## Estimasi Model

### Pooled OLS (Baseline)

```stata
reg ln_wage age c.age#c.age tenure hours i.race, robust
estimates store pooled
```

### Fixed Effects (FE)

```stata
xtreg ln_wage age c.age#c.age tenure hours, fe robust
estimates store fe
```

> **Catatan:** Variabel time-invariant (seperti `race`) otomatis tereliminasi di FE.

### Random Effects (RE)

```stata
xtreg ln_wage age c.age#c.age tenure hours i.race, re robust
estimates store re
```

### Perbandingan Output

```stata
estimates table pooled fe re, star stats(N r2 r2_a r2_w r2_b r2_o)
```

## Hausman Test: FE vs RE

Hipotesis:
- $H_0$: RE konsisten dan efisien (gunakan RE)
- $H_1$: RE tidak konsisten (gunakan FE)

```stata
* Estimasi ulang tanpa robust untuk Hausman test
quietly xtreg ln_wage age c.age#c.age tenure hours, fe
estimates store fe_haus

quietly xtreg ln_wage age c.age#c.age tenure hours, re
estimates store re_haus

hausman fe_haus re_haus
```

> Jika $p < 0.05$, tolak $H_0$ → gunakan **Fixed Effects**.

## Uji Asumsi

### 1. Heteroskedastisitas (Modified Wald Test)

```stata
* Setelah xtreg, fe
quietly xtreg ln_wage age c.age#c.age tenure hours, fe
xttest3
```

### 2. Autokorelasi (Wooldridge Test)

```stata
xtserial ln_wage age tenure hours
```

### 3. Cross-Sectional Dependence (Pesaran CD Test)

```stata
quietly xtreg ln_wage age c.age#c.age tenure hours, fe
xtcsd, pesaran abs
```

### Solusi jika Asumsi Dilanggar

| Masalah | Solusi |
|---------|--------|
| Heteroskedastisitas | `robust` atau `vce(cluster idcode)` |
| Autokorelasi | `vce(cluster idcode)` |
| Keduanya | Driscoll-Kraay: `xtscc` |

```stata
* Cluster-robust standard errors (solusi umum)
xtreg ln_wage age c.age#c.age tenure hours, fe vce(cluster idcode)
```

## Visualisasi

```stata
* Marginal effects of age (quadratic)
quietly xtreg ln_wage c.age##c.age tenure hours, fe robust
margins, at(age=(18(2)46)) vsquish
marginsplot, ///
    title("Predicted Log Wage by Age") ///
    ytitle("Predicted ln(wage)") ///
    xtitle("Age") ///
    scheme(s2color)
```

## Ringkasan Perintah

| Langkah | Perintah STATA | Keterangan |
|---------|---------------|------------|
| Setup panel | `xtset id time` | Deklarasi struktur panel |
| Deskriptif | `xtsum`, `xtdescribe` | Statistik between/within |
| Fixed Effects | `xtreg y x, fe` | Eliminasi efek individu |
| Random Effects | `xtreg y x, re` | Efek individu sebagai random |
| Hausman test | `hausman fe re` | Pilih FE vs RE |
| Heteroskedastisitas | `xttest3` | Modified Wald test |
| Autokorelasi | `xtserial` | Wooldridge test |
| Robust SE | `vce(cluster id)` | Cluster-robust |

### Referensi

- Baltagi, B. H. (2021). *Econometric Analysis of Panel Data.* 6th ed. Springer.
- Wooldridge, J. M. (2010). *Econometric Analysis of Cross Section and Panel Data.* 2nd ed. MIT Press.

---

*Tutorial selanjutnya: Dynamic Panel Data — GMM Estimation di STATA.*